In [14]:
import json
from collections import defaultdict

IOU_THRESHOLD = 0.5


def span_iou(span1, span2):
    s1, e1 = span1["start"], span1["end"]
    s2, e2 = span2["start"], span2["end"]

    inter = max(0, min(e1, e2) - max(s1, s2))
    union = max(e1, e2) - min(s1, s2)

    if union == 0:
        return 0
    return inter / union


def deduplicate_spans(spans):
    seen = set()
    cleaned = []
    for s in spans:
        key = (s["start"], s["end"], s.get("text", ""))
        if key not in seen:
            cleaned.append(s)
            seen.add(key)
    return cleaned


def match_spans(spans_a, spans_b):
    matches = []
    used_b = set()
    for i, sa in enumerate(spans_a):
        best_match = None
        best_iou = 0
        for j, sb in enumerate(spans_b):
            if j in used_b:
                continue
            iou = span_iou(sa, sb)
            if iou > best_iou:
                best_iou = iou
                best_match = j
        if best_iou >= IOU_THRESHOLD:
            matches.append((i, best_match))
            used_b.add(best_match)
    return matches


def compute_metrics(spans_a, spans_b, labels_a=None, labels_b=None):
    matches = match_spans(spans_a, spans_b)
    tp = len(matches)
    precision = tp / len(spans_a) if spans_a else 0
    recall = tp / len(spans_b) if spans_b else 0
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    label_agree = None
    if labels_a and labels_b:
        if isinstance(labels_a, str):
            labels_a = [labels_a]
        if isinstance(labels_b, str):
            labels_b = [labels_b]
        agree = 0
        for i, j in matches:
            try:
                if labels_a[i] == labels_b[j]:
                    agree += 1
            except Exception as e:
                pass
        label_agree = agree / tp if tp else 0
    return precision, recall, f1, label_agree


def group_by_review(data):
    grouped = defaultdict(list)
    for item in data:
        grouped[item["review_id"]].append(item)
    return grouped


def evaluate_dataset(data):

    grouped = group_by_review(data)

    desc_scores = []
    review_scores = []

    for review_id, annots in grouped.items():

        if len(annots) < 2:
            continue

        a, b = annots[0], annots[1]

        desc_a = deduplicate_spans(a.get("desc_attr_span", []))
        desc_b = deduplicate_spans(b.get("desc_attr_span", []))

        desc_labels_a = a.get("desc_attr_name", [])
        desc_labels_b = b.get("desc_attr_name", [])

        p, r, f1, la = compute_metrics(desc_a, desc_b, desc_labels_a, desc_labels_b)

        desc_scores.append((p, r, f1, la))

        review_a = deduplicate_spans(a.get("review_attr_span", []))
        review_b = deduplicate_spans(b.get("review_attr_span", []))

        review_labels_a = a.get("review_attr_name", [])
        review_labels_b = b.get("review_attr_name", [])

        p, r, f1, la = compute_metrics(review_a, review_b, review_labels_a, review_labels_b)

        review_scores.append((p, r, f1, la))

    return desc_scores, review_scores


def average_scores(scores):

    if not scores:
        return 0,0,0,0

    p = sum(s[0] for s in scores) / len(scores)
    r = sum(s[1] for s in scores) / len(scores)
    f = sum(s[2] for s in scores) / len(scores)

    la = [s[3] for s in scores if s[3] is not None]
    la = sum(la) / len(la) if la else None

    return p,r,f,la


if __name__ == "__main__":

    with open("/home/leah/MDSCL/COLX523/COLX523_Freya_Leah_Wei_Yirui/Sprint_3/data/annotation_final/pair_1.json") as f:
        data = json.load(f)

    desc_scores, review_scores = evaluate_dataset(data)

    p,r,f,la = average_scores(desc_scores)

    print("\nDESC ATTRIBUTE IAA")
    print("Precision:", round(p,3))
    print("Recall:", round(r,3))
    print("F1:", round(f,3))
    print("Label Agreement:", round(la,3) if la else None)

    p,r,f,la = average_scores(review_scores)

    print("\nREVIEW ATTRIBUTE IAA")
    print("Precision:", round(p,3))
    print("Recall:", round(r,3))
    print("F1:", round(f,3))
    print("Label Agreement:", round(la,3) if la else None)



DESC ATTRIBUTE IAA
Precision: 0.899
Recall: 0.89
F1: 0.891
Label Agreement: 0.669

REVIEW ATTRIBUTE IAA
Precision: 0.959
Recall: 0.958
F1: 0.957
Label Agreement: 0.614
